In [29]:
import os
import openai

from langsmith import Client
from qdrant_client import QdrantClient

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

#### Download an example reference data point from LangSmith

In [30]:
client = Client()

In [31]:
dataset = client.read_dataset(dataset_name="rag-evaluation-dataset")

In [32]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('c6fc2f01-7608-4a72-b540-3c6bee39d352'), created_at=datetime.datetime(2026, 5, 16, 15, 22, 45, 600983, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 5, 16, 15, 22, 45, 600983, tzinfo=TzInfo(0)), example_count=36, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.22000-SP0', 'sdk_version': '0.8.5', 'runtime_version': '3.12.13', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [33]:
list(client.list_examples(dataset_id=dataset.id,limit = 10))[0].outputs

{'ground_truth': 'The YYJOY Spy Camera (B0C4P45FXV) supports recording onto a MicroSD card (not included). The AOSU WirelessCam Pro (B09X7D59WX) includes built-in 32GB memory on the base for local storage. The Light Bulb Security Camera (B0BQHMFG85) mentions built-in SD/TF memory card support as well as cloud storage options.',
 'reference_context_ids': ['B0C4P45FXV', 'B09X7D59WX', 'B0BQHMFG85'],
 'reference_descriptions': ["YYJOY Spy Camera Hidden Camera Wireless Small Camera Nanny Cam,Wi-Fi,Night Vision,4K HD Mini Security Camera Indoor,Phone APP, Motion Detection,Battery Powered Spy Cam for Home,Easy to Use 𝐂𝐨𝐦𝐩𝐚𝐜𝐭 𝐚𝐧𝐝 𝐄𝐚𝐬𝐲 𝐭𝐨 𝐜𝐨𝐧𝐧𝐞𝐜𝐭: This camera is easy to install with magnetic base. The wifi connection is optimized for faster and more convenient scanning, and the app supports up to 4 cameras, making it perfect for home security or monitoring. 𝐖𝐢𝐫𝐞𝐥𝐞𝐬𝐬 𝐋𝐨𝐧𝐠-𝐥𝐚𝐬𝐭𝐢𝐧𝐠 𝐁𝐚𝐭𝐭𝐞𝐫𝐲 𝐋𝐢𝐟𝐞: Equipped with a 1200 mAh rechargeable battery, this wireless indoor camera can record continuously for 

In [34]:
list(client.list_examples(dataset_id=dataset.id,limit = 10))[0].inputs

{'question': 'Which items in the chunks explicitly require or include a MicroSD or local storage option?'}

In [35]:
reference_input = list(client.list_examples(dataset_id=dataset.id,limit = 10))[0].inputs
reference_output = list(client.list_examples(dataset_id=dataset.id,limit = 10))[0].outputs

In [47]:
import openai
from qdrant_client import QdrantClient



def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model,
    )

    return response.data[0].embedding


def retrieve_data(query, qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_context.append(point.payload["description"])
        similarity_scores.append(point.score)
        retrieved_context_ratings.append(point.payload["average_rating"])
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"-ID: {id}, Rating: {rating}, Description: {chunk}\n"

    return formatted_context



def build_prompt(preprocessed_context, question):
    prompt = f"""
you are a shopping assistant that can answer questions about products in stock.

You will be given a question and a list of context. 

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as avaialable products.

Context:
{preprocessed_context}

Question: {question}
"""
    return prompt


def generate_answer(prompt):
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning_effort="minimal"
        # temperature=0.7,
        # max_tokens=500,
    )

    return response.choices[0].message.content



def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(
        url="http://localhost:6333")
    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    final_result = {
        "answer": answer,
        "question": question,
        "retrieved_context": retrieved_context["retrieved_context"],
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result


In [48]:
rag_pipeline("Do you have a charger?",top_k=5)

{'answer': 'Yes. We have several charger options in available products:\n\n- B0BHRHS1Q7: Bauhoo 3-in-1 Fast Wireless Charger (foldable) for iPhone, Apple Watch, AirPods.\n- B0BDRGBRN5: New Lenovo 230W AC Adapter Charger (UL listed; compatible with many Lenovo models).\n- B008FPY1VG: 65W Replacement AC Adapter Charger for various HP notebooks (compatible with many HP models).\n\nIf you’d like a specific compatibility (brand/model) or more details, tell me your device and I’ll help you pick the best match.',
 'question': 'Do you have a charger?',
 'retrieved_context': ['Bauhoo Charging Station for Apple Multiple Devices, 3 in 1 Fast Wireless Charger Foldable iPhone 14/13/12/11/Pro/XS/Xs Max/XR/X/SE/8/8 Plus Watch 8/7/6/SE/5/4/3/2 AirPods 3/2/Pro with Adapter Pink 0',
  'New Lenovo 230w ac Adapter Compatible with Lenovo Legion 5 Charger for Lenovo Legion 5 7 5P C7 S7 Y540 Y545 Y740 Y730 Y900 Y910 Y920 Y7000 4X20E75111 GX20L29347 ADL230NDC3A ADL230NLC3A 💯Output: 20V 11.5A 230W Input:100V-2

### RAGAS Metrics

In [38]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

C:\Users\bolly\AppData\Local\Temp\ipykernel_3392\3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\bolly\AppData\Local\Temp\ipykernel_3392\3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\bolly\AppData\Local\Temp\ipykernel_3392\3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'r

In [39]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


C:\Users\bolly\AppData\Local\Temp\ipykernel_3392\3259588242.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
C:\Users\bolly\AppData\Local\Temp\ipykernel_3392\3259588242.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [40]:
reference_input, reference_output

({'question': 'Which items in the chunks explicitly require or include a MicroSD or local storage option?'},
 {'ground_truth': 'The YYJOY Spy Camera (B0C4P45FXV) supports recording onto a MicroSD card (not included). The AOSU WirelessCam Pro (B09X7D59WX) includes built-in 32GB memory on the base for local storage. The Light Bulb Security Camera (B0BQHMFG85) mentions built-in SD/TF memory card support as well as cloud storage options.',
  'reference_context_ids': ['B0C4P45FXV', 'B09X7D59WX', 'B0BQHMFG85'],
  'reference_descriptions': ["YYJOY Spy Camera Hidden Camera Wireless Small Camera Nanny Cam,Wi-Fi,Night Vision,4K HD Mini Security Camera Indoor,Phone APP, Motion Detection,Battery Powered Spy Cam for Home,Easy to Use 𝐂𝐨𝐦𝐩𝐚𝐜𝐭 𝐚𝐧𝐝 𝐄𝐚𝐬𝐲 𝐭𝐨 𝐜𝐨𝐧𝐧𝐞𝐜𝐭: This camera is easy to install with magnetic base. The wifi connection is optimized for faster and more convenient scanning, and the app supports up to 4 cameras, making it perfect for home security or monitoring. 𝐖𝐢𝐫𝐞𝐥𝐞𝐬𝐬 𝐋𝐨𝐧𝐠-𝐥𝐚𝐬𝐭𝐢𝐧𝐠 𝐁𝐚𝐭𝐭𝐞

In [49]:
result = rag_pipeline(reference_input["question"])

In [50]:
result

{'answer': 'From the available products, the item that explicitly includes or references local storage is:\n\n- ID B0BJV9TZZ8: Mini Voice Activated Recorder with 64GB memory (local storage) and USB drive functionality.\n\nNo other items in the available products mention MicroSD or local storage options.',
 'question': 'Which items in the chunks explicitly require or include a MicroSD or local storage option?',
 'retrieved_context': ['Mini Voice Activated Recorder,64GB Taheng Small Recording Device with 750 Hrs Recording Capacity,USB Digital Audio Recorder for Lecture Interview Meeting Class 【64GB Memory & Long-Lasting Battery】The mini usb voice activated recorder come with 64GB memory. Enough space for more than 25 hours continuous recording and store up to 750 hours recording files. It can also be a professional MP3 player and USB Drive. Our 64GB portable recording device have unique designed makes it look unlike a audio recorder at all!!! 【Mini Size & One-Key Recording】Ultra-small si

In [ ]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run.inputs["question"],
        response=run.outputs["answer"],
        retrieved_contexts=run["retrieved_context"],
        )
    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [52]:
await ragas_faithfulness(result, "")

0.3333333333333333

In [53]:
async def ragas_response_relevanacy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
        )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [54]:
await ragas_response_relevanacy(result, "")

np.float64(0.6396205899236821)

In [61]:
async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
        )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [62]:
await ragas_context_precision_id_based(result, reference_output)

0.0

In [63]:
async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
        )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [64]:
await ragas_context_recall_id_based(result, reference_output)

0.0